In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_G.xpt"
csv_file = DATA_DIR / "HIQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 HIQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HIQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9756:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 17:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,756 rows x 17 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached HIQ_G.xpt:
#
#     tiny artifact count = 0
#
# Therefore NO zero restoration is needed.
#
# IMPORTANT:
#
# If a different/local copy produces tiny values,
# STOP rather than automatically replacing them.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HIQ_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total > 0:

    print(
        "\nTrue zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected numeric zeros found "
        "in HIQ_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No ordinary numeric zeros."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Always do this BEFORE converting anything to Int64.
#
# Attached HIQ_G contains:
#
#     genuine fractional variables = NONE
#
# All fields are identifiers or questionnaire codes.
#
# If a fractional value unexpectedly appears,
# STOP rather than rounding it.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in HIQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# HIQ_G contains NO genuine decimal fields.
#
# Therefore all 17 variables can safely become
# nullable Int64.
#
# IMPORTANT:
#
# NO .round() IS USED FOR CONVERSION.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 17 HIQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 8. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 9. CDC CHECK - HIQ011
# ============================================================

#
# Covered by health insurance
#
# CDC:
#
# 1 = Yes          8,033
# 2 = No           1,703
# 7 = Refused          4
# 9 = Don't know      16
# Missing              0
#

check_frequency(
    "HIQ011",
    {
        1: 8033,
        2: 1703,
        7: 4,
        9: 16
    },
    0
)


# ============================================================
# 10. CDC CHECK - HIQ031A PRIVATE INSURANCE
# ============================================================

#
# IMPORTANT:
#
# HIQ031A-HIQ031J are checkbox-style insurance
# type variables.
#
# They do NOT use simple 1/2 yes/no codes.
#
# HIQ031A:
#
# 14 = Covered by private insurance
# 77 = Refused
# 99 = Don't know
#

check_frequency(
    "HIQ031A",
    {
        14: 4357,
        77: 1,
        99: 26
    },
    5372
)


# ============================================================
# 11. CDC CHECK - HIQ031B MEDICARE
# ============================================================

check_frequency(
    "HIQ031B",
    {
        15: 1206
    },
    8550
)


# ============================================================
# 12. CDC CHECK - HIQ031C MEDI-GAP
# ============================================================

check_frequency(
    "HIQ031C",
    {
        16: 14
    },
    9742
)


# ============================================================
# 13. CDC CHECK - HIQ031D MEDICAID
# ============================================================

check_frequency(
    "HIQ031D",
    {
        17: 2175
    },
    7581
)


# ============================================================
# 14. CDC CHECK - HIQ031E SCHIP
# ============================================================

check_frequency(
    "HIQ031E",
    {
        18: 98
    },
    9658
)


# ============================================================
# 15. CDC CHECK - HIQ031F MILITARY HEALTH CARE
# ============================================================

check_frequency(
    "HIQ031F",
    {
        19: 162
    },
    9594
)


# ============================================================
# 16. CDC CHECK - HIQ031G INDIAN HEALTH SERVICE
# ============================================================

check_frequency(
    "HIQ031G",
    {
        20: 11
    },
    9745
)


# ============================================================
# 17. CDC CHECK - HIQ031H STATE PLAN
# ============================================================

check_frequency(
    "HIQ031H",
    {
        21: 592
    },
    9164
)


# ============================================================
# 18. CDC CHECK - HIQ031I OTHER GOVERNMENT
# ============================================================

check_frequency(
    "HIQ031I",
    {
        22: 299
    },
    9457
)


# ============================================================
# 19. CDC CHECK - HIQ031J SINGLE SERVICE PLAN
# ============================================================

check_frequency(
    "HIQ031J",
    {
        23: 119
    },
    9637
)


# ============================================================
# 20. CDC CHECK - HIQ031AA NO COVERAGE TYPE
# ============================================================

#
# 40 = No coverage of any type
#
# Note:
#
# This is different from HIQ011=2.
#
# HIQ011 asks whether participant has coverage.
# HIQ031AA is part of the coverage-type selection set.
#

check_frequency(
    "HIQ031AA",
    {
        40: 3
    },
    9753
)


# ============================================================
# 21. CDC CHECK - HIQ260 MEDICARE
# ============================================================

#
# Asked based on age/insurance routing.
#
# CDC:
#
# 1 Yes        132
# 2 No         112
# 7 Refused      3
# 9 Don't know   0
# Missing     9,509
#

check_frequency(
    "HIQ260",
    {
        1: 132,
        2: 112,
        7: 3
    },
    9509
)


# ============================================================
# 22. CDC CHECK - HIQ105 INSURANCE CARD
# ============================================================

check_frequency(
    "HIQ105",
    {
        1: 767,
        2: 88
    },
    8901
)


# ============================================================
# 23. CDC CHECK - HIQ270 PRESCRIPTION COVERAGE
# ============================================================

#
# CDC:
#
# 1 Yes          7,596
# 2 No             381
# 7 Refused          3
# 9 Don't know      66
# Missing        1,710
#

check_frequency(
    "HIQ270",
    {
        1: 7596,
        2: 381,
        7: 3,
        9: 66
    },
    1710
)


# ============================================================
# 24. CDC CHECK - HIQ210 GAP IN COVERAGE
# ============================================================

#
# Time in past 12 months with no insurance coverage
#
# CDC:
#
# 1 Yes            509
# 2 No           7,533
# 7 Refused          0
# 9 Don't know       4
# Missing        1,710
#

check_frequency(
    "HIQ210",
    {
        1: 509,
        2: 7533,
        9: 4
    },
    1710
)


# ============================================================
# 25. COVERAGE TYPE ROUTING CONSISTENCY CHECK
# ============================================================

#
# Coverage-type variables should only be populated for
# respondents whose HIQ011 response indicates coverage.
#
# Attached file:
#
# no nonmissing HIQ031* type variable occurs outside
# HIQ011 == 1.
#

COVERAGE_TYPE_COLUMNS = [
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA"
]


print("\n--- COVERAGE TYPE ROUTING CHECK ---")


for col in COVERAGE_TYPE_COLUMNS:

    invalid_count = int(
        (
            df[col].notna()
            &
            ~df["HIQ011"].eq(1)
        )
        .sum()
    )


    if invalid_count != 0:

        raise ValueError(
            f"{col}: {invalid_count} populated values "
            "found outside HIQ011=1."
        )


    print(
        f"PASS: {col}"
    )


print(
    "PASS: All HIQ031* coverage-type values "
    "follow the expected routing pattern."
)


# ============================================================
# 26. HIQ270 / HIQ210 ELIGIBILITY CONSISTENCY
# ============================================================

#
# In the attached release:
#
# HIQ270 and HIQ210 each contain:
#
#     8,046 non-missing
#     1,710 missing
#
# Their missing-position masks are exactly the same.
#

hiq270_nonmissing = int(
    df["HIQ270"]
    .notna()
    .sum()
)


hiq210_nonmissing = int(
    df["HIQ210"]
    .notna()
    .sum()
)


if hiq270_nonmissing != 8046:

    raise ValueError(
        "Unexpected HIQ270 non-missing count."
    )


if hiq210_nonmissing != 8046:

    raise ValueError(
        "Unexpected HIQ210 non-missing count."
    )


same_missing_pattern = np.array_equal(
    df["HIQ270"]
    .isna()
    .to_numpy(),

    df["HIQ210"]
    .isna()
    .to_numpy()
)


if not same_missing_pattern:

    raise ValueError(
        "HIQ270 and HIQ210 missing-position "
        "patterns differ unexpectedly."
    )


print(
    "PASS: HIQ270 and HIQ210 eligibility/missing "
    "patterns are internally consistent."
)


# ============================================================
# 27. ALLOWED RELEASED CODE CHECK
# ============================================================

#
# Additional protection against unexpected values.
#

ALLOWED_VALUES = {

    "HIQ011": {
        1,
        2,
        7,
        9
    },

    "HIQ031A": {
        14,
        77,
        99
    },

    "HIQ031B": {
        15
    },

    "HIQ031C": {
        16
    },

    "HIQ031D": {
        17
    },

    "HIQ031E": {
        18
    },

    "HIQ031F": {
        19
    },

    "HIQ031G": {
        20
    },

    "HIQ031H": {
        21
    },

    "HIQ031I": {
        22
    },

    "HIQ031J": {
        23
    },

    "HIQ031AA": {
        40
    },

    "HIQ260": {
        1,
        2,
        7
    },

    "HIQ105": {
        1,
        2
    },

    "HIQ270": {
        1,
        2,
        7,
        9
    },

    "HIQ210": {
        1,
        2,
        9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected released codes: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only "
        "expected released codes."
    )


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HIQ_G requires ZERO numeric corrections.
#
# Therefore every numeric value must remain exactly
# equal to the source XPT.
#
# Only CSV textual representation changes:
#
#     SEQN
#     62161.0
#        ->
#     62161
#
#     HIQ011
#     1.0
#      ->
#     1
#
#     HIQ031A
#     14.0
#       ->
#     14
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 17 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# No rounding.
#
# No:
#
#     float_format=
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 17 CSV columns round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 32. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 33. READ RAW CSV AS TEXT
# ============================================================

#
# Numeric fidelity was checked above.
#
# Now inspect the actual exported formatting.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 35. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# HIQ_G contains NO genuine decimal variables.
#
# Therefore ".0" should not occur in any populated
# value in this particular file.
#
# This rule is safe for HIQ_G because ALL 17 fields
# are proven whole-number identifiers/codes.
#
# DO NOT apply this global rule to Dietary files or
# Lab/Exam/Questionnaire files containing genuine decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 36. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 37. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 38. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "HIQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original true numeric zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 HIQ_G XPT -> CSV
Rows: 9,756
Columns: 17
PASS: Structure = 9,756 rows x 17 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 0
PASS: No tiny XPORT artifacts.
PASS: No ordinary numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HIQ011: 0 fractional observations
HIQ031A: 0 fractional observations
HIQ031B: 0 fractional observations
HIQ031C: 0 fractional observations
HIQ031D: 0 fractional observations
HIQ031E: 0 fractional observations
HIQ031F: 0 fractional observations
HIQ031G: 0 fractional observations
HIQ031H: 0 fractional observations
HIQ031I: 0 fractional observations
HIQ031J: 0 fractional observations
HIQ031AA: 0 fractional observations
HIQ260: 0 fractional observations
HIQ105: 0 fractional observations
HIQ270: 0 fractional observations
HIQ210: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

P